In [2]:
import laspy
import pandas as pd
import numpy as np
from jakteristics import compute_features
# load the classification function
from sklearn.naive_bayes import GaussianNB
from sklearn.feature_selection import SequentialFeatureSelector
# for n__radius in [0.5, 1, 1.2, 3]
n_radius = 1

# Add Dimension Function
def add_dimension(las_file, dimension_name, dtype):
    """  This function is used to add new dimension to the point cloud """ 
    if dtype == 'float':        
        las_file.add_extra_dim(laspy.ExtraBytesParams(
            name=dimension_name,
            type=np.float64,
            description=dimension_name
        )) 
    elif dtype == 'int':
        las_file.add_extra_dim(laspy.ExtraBytesParams(
            name=dimension_name,
            type=np.int32,
            description=dimension_name
        ))                     
    return las_file

# Normalization Function
def normalize_feature(arr):
    min_val = np.min(arr)
    max_val = np.max(arr)
    return (arr - min_val) / (max_val - min_val) if max_val > min_val else arr * 0


# Read Point Cloud function
def read_point_cloud(las_path):
    my_point_cloud = laspy.read(las_path)
    offsets = my_point_cloud.header.offsets
    scales = my_point_cloud.header.scales
    x = my_point_cloud['X']*scales[0] + offsets[0]
    y = my_point_cloud['Y']*scales[1] + offsets[1]
    z = my_point_cloud['Z']*scales[2] + offsets[2]
    #######################
    point_cloud_data_frame = pd.DataFrame()
    point_cloud_data_frame['x']  = x
    point_cloud_data_frame['y']  = y
    point_cloud_data_frame['z']  = z
    dimension_names = list(my_point_cloud.point_format.dimension_names)
    for i, dimension in enumerate(dimension_names):
        point_cloud_data_frame[dimension] = np.array(getattr(my_point_cloud, dimension))
    return point_cloud_data_frame, my_point_cloud, offsets, scales





In [3]:
import numpy as np

# Sequential Feature Extractor (feature builder)
def extract_feature(point_cloud_df, n_radius, add_extra=("intensity", "return_number", "scan_angle_rank")):
    selected_geometric_features = [
        "eigenvalue_sum", "omnivariance", "eigenentropy", "anisotropy", "planarity", "linearity",
        "PCA1", "PCA2", "surface_variation", "sphericity", "verticality",
        "nx", "ny", "nz", "number_of_neighbors"
    ]

    # xyz as numpy
    xyz = point_cloud_df[["x", "y", "z"]].to_numpy(dtype=float)

    # compute geometric features: (N, G)
    geometric_features = compute_features(
        xyz,
        search_radius=n_radius,
        feature_names=selected_geometric_features
    )
    geometric_features = np.asarray(geometric_features, dtype=float)
    geometric_features[np.isnan(geometric_features)] = 0.0

    # build output feature list
    feature_names = list(selected_geometric_features)

    extra_arrays = []
    for col in add_extra:
        if col in point_cloud_df.columns:
            arr = point_cloud_df[col].to_numpy(dtype=float).reshape(-1, 1)
            arr[np.isnan(arr)] = 0.0
            extra_arrays.append(arr)
            feature_names.append(col)

    # final X
    if extra_arrays:
        X = np.hstack([geometric_features] + extra_arrays)
    else:
        X = geometric_features

    return X, feature_names

# =========================
# Read point clouds
# =========================
trees_df, _, _, _ = read_point_cloud(r"train\Trees.las")
ground_df, _, _, _ = read_point_cloud(r"train\Ground.las")
cars_df, _, _, _ = read_point_cloud(r"train\Cars.las")
buildings_df, _, _, _ = read_point_cloud(r"train\Buildings.las")


# =========================
# Extract features
# =========================
X_tree, feature_names = extract_feature(trees_df, n_radius)
X_ground, _ = extract_feature(ground_df, n_radius)
X_car, _ = extract_feature(cars_df, n_radius)
X_building, _ = extract_feature(buildings_df, n_radius)


# =========================
# Generate labels
# =========================
y_tree = np.ones(len(X_tree)) * 1        # trees
y_ground = np.ones(len(X_ground)) * 0    # ground
y_car = np.ones(len(X_car)) * 2          # cars
y_building = np.ones(len(X_building)) * 3 # buildings


# =========================
# Combine dataset
# =========================
X_all = np.vstack([
    X_tree,
    X_ground,
    X_car,
    X_building
])

y_all = np.concatenate([
    y_tree,
    y_ground,
    y_car,
    y_building
])


print("Training dataset shape:", X_all.shape)
print("Labels shape:", y_all.shape)


Training dataset shape: (148837, 18)
Labels shape: (148837,)


In [4]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
# =========================
# Read point clouds (TEST)
# =========================
trees_df_test, _, _, _ = read_point_cloud(r"test\Trees.las")
ground_df_test, _, _, _ = read_point_cloud(r"test\Ground.las")
cars_df_test, _, _, _ = read_point_cloud(r"test\Cars.las")
buildings_df_test, _, _, _ = read_point_cloud(r"test\Buildings.las")


# =========================
# Extract features
# =========================
X_tree_test, _ = extract_feature(trees_df_test, n_radius)
X_ground_test, _ = extract_feature(ground_df_test, n_radius)
X_car_test, _ = extract_feature(cars_df_test, n_radius)
X_building_test, _ = extract_feature(buildings_df_test, n_radius)


# =========================
# Generate labels
# =========================
y_tree_test = np.ones(len(X_tree_test)) * 1        # trees
y_ground_test = np.ones(len(X_ground_test)) * 0    # ground
y_car_test = np.ones(len(X_car_test)) * 2          # cars
y_building_test = np.ones(len(X_building_test)) * 3 # buildings


# =========================
# Combine dataset
# =========================
X_test = np.vstack([
    X_tree_test,
    X_ground_test,
    X_car_test,
    X_building_test
])

y_test = np.concatenate([
    y_tree_test,
    y_ground_test,
    y_car_test,
    y_building_test
])




In [64]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_all,
    y_all,
    test_size=0.3,
    stratify=y_all,
    random_state=42
)

In [5]:


'''
In the class we didn't talk about PCA or sequential feature selection'
You may use PCA on the train Data here. For the test and all data, you may use the fitted PCA to training data on test and all data before prediction.
'''
# 5) SFS + classifier (Pipeline prevents leakage)
pipe = Pipeline([
     ("scaler", StandardScaler()),
    ("sfs", SequentialFeatureSelector(
        estimator=GaussianNB(),
        n_features_to_select=6,
        direction="forward",
        scoring="accuracy",
        cv=5,
        n_jobs=-1
    )),
    ("clf", GaussianNB())
])

pipe.fit(X_all, y_all)

mask = pipe.named_steps["sfs"].get_support()



In [ ]:
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, cohen_kappa_score, f1_score
selected_features = np.array(feature_names)[mask]
print("Selected features:", selected_features)

y_pred = pipe.predict(X_test_all)
print("Accuracy:", accuracy_score(y_test_all, y_pred))
print(classification_report(y_test_all, y_pred))
cm = confusion_matrix(y_test_all, y_pred, labels=[0,1,2,3])

kappa = cohen_kappa_score(y_test_all, y_pred)
f1m = f1_score(y_test_all, y_pred, average="macro")
print(, kappa, f1m)


Selected features: ['PCA1' 'sphericity' 'verticality' 'nx' 'nz' 'intensity']
Accuracy: 0.6077176246664892


c:\Users\quetr\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Users\quetr\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Users\quetr\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


              precision    recall  f1-score   support

         0.0       0.68      0.83      0.75    111130
         1.0       0.41      0.32      0.36     31598
         2.0       0.00      0.00      0.00       780
         3.0       0.31      0.15      0.20     33023

    accuracy                           0.61    176531
   macro avg       0.35      0.32      0.33    176531
weighted avg       0.56      0.61      0.57    176531

[[92403 10029     0  8698]
 [19542  9984     0  2072]
 [  538   209     0    33]
 [23939  4190     0  4894]] 0.16911115636042928 0.32598623696735424


In [88]:
pipe.steps[2][1].get_params()

{'priors': None, 'var_smoothing': 1e-09}

In [ ]:
import time
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, cohen_kappa_score, f1_score
results = []
from xgboost import XGBClassifier
def evaluate_model(name, strategy, pipe, Xtr, ytr, Xte, yte):
    print("training")
    # train time
    t0 = time.perf_counter()
    pipe.fit(Xtr, ytr)
    train_time = time.perf_counter() - t0
    print("testing")
    # test time
    t1 = time.perf_counter()
    y_pred = pipe.predict(Xte)
    test_time = time.perf_counter() - t1

    cm = confusion_matrix(yte, y_pred, labels=[0,1,2,3])
    acc = accuracy_score(yte, y_pred)
    kappa = cohen_kappa_score(yte, y_pred)
    f1m = f1_score(yte, y_pred, average="macro")

    # hyperparameters (even if default)
    params = pipe.get_params(deep=False)

    row = {
        "Strategy": strategy,
        "Classifier": name,
        "TrainTime_s": train_time,
        "TestTime_s": test_time,
        "Accuracy": acc,
        "Kappa": kappa,
        "F1_macro": f1m,
        "ConfusionMatrix": cm,
        "Hyperparams": params
    }
    return row, y_pred, pipe
classifiers = {
    "SVM_rbf":XGBClassifier(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.01,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="multi:softmax",
        num_class=4,
        tree_method="hist",
        eval_metric="mlogloss",
        random_state=42
    )
}
sfs_selected_feature_names_by_model = {}

for clf_name, clf in classifiers.items():
    pipe_sfs = Pipeline([
        ("scaler", StandardScaler()),
        ("sfs", SequentialFeatureSelector(
            estimator=clf,        # selector estimator (fast)
            n_features_to_select=5,
            direction="forward",
            scoring="accuracy",
            cv=5,
            n_jobs=-1
        )),
        ("clf", clf)
    ])

    row, y_pred, fitted_pipe = evaluate_model(
        clf_name, f"SFS_forward(k=6)", pipe_sfs, X_all, y_all, X_test, y_test
    )
    results.append(row)

    # report selected features for this fitted pipeline
    mask = fitted_pipe.named_steps["sfs"].get_support()
    sfs_selected_feature_names_by_model[clf_name] = list(np.array(feature_names)[mask])
    print(row)

print("Selected features (SFS):")
for k, v in sfs_selected_feature_names_by_model.items():
    print(k, "=>", v)

training
